# Lesson 720.24: Which Edits Preserve Balance? Workbench

## Counterstate inspection

Preparation: select cached binaries, inspect subprocess stderr, record predicted
booleans, evaluate immutable profiles, distinguish instance observations from
quantified statements. JSON packets expose multiplicity storage explicitly.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

assert sys.version_info >= (3, 10)
assert shutil.which("cargo"), "Cargo must be installed"
work = Path.cwd().resolve()
repo = next(
    p
    for p in [work, *work.parents]
    if (p / "omega_engine/crates_rs/omega_wdp/Cargo.toml").is_file()
)
cache = Path(os.environ["CARGO_TARGET_DIR"]).resolve()
assert repo not in cache.parents and cache != repo, "Use an external cache"
assert (repo / "omega_engine/crates_rs/Cargo.lock").is_file()

In [ ]:
prediction = {
    "add_term": True,
    "add_forward": False,
    "add_reverse": True,
    "remove_forward": False,
    "identify_touching": False,
    "fill_three_faces": False,
    "fill_seven_faces": True,
}
prediction_explanation = "The quantified context rule can fail although one zero input survives."
assert len(prediction) == 7

Submission: invoke the example executable through Cargo's locked offline mode.
When troubleshooting, retain compiler diagnostics, resolved workspace location,
exit status, stdout schema, and dependency-cache configuration.


In [ ]:
command = [
    "cargo",
    "run",
    "--quiet",
    "--offline",
    "--locked",
    "-j1",
    "--target-dir",
    str(cache),
    "-p",
    "omega_wdp",
    "--example",
    "balance_preserving_edits",
]
result = subprocess.run(
    command, cwd=repo / "omega_engine/crates_rs", capture_output=True, text=True, check=False
)
assert result.returncode == 0, result.stderr
data = json.loads(result.stdout)
cases = {item["name"]: item for item in data["cases"]}
assert set(cases) == set(prediction)

In [ ]:
observed = {name: item["universal_preserves"] for name, item in cases.items()}
assert observed == prediction, (prediction, observed)
assert data["disabled_fill_refused"], "An already filled face must be rejected"
assert data["authority_effect"] == "None"
assert not cases["fill_three_faces"]["actual_after_balanced"]
assert cases["fill_seven_faces"]["actual_after_balanced"]

Inspection: compare observed values against predictions. Locate the submitted
packet separately from its adversarial replacement. Report discrepancies using
the case identifier, arc orientation, refusal flag, and witness magnitude.
Rerun only after changing either executable inputs or predictions.


In [ ]:
merge = cases["identify_touching"]
assert merge["actual_before_balanced"] and merge["actual_after_balanced"]
assert not merge["universal_preserves"]
assert merge["counterexample"]["value"] != "0"
assert merge["counterexample"]["source"] != merge["source"]
assert not cases["add_forward"]["actual_after_balanced"]
print(observed)
print("The successful zero merge coexists with a separately checked negative witness.")